In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/retrieval-rag/rag-from-scratch/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 04 · Reranking

First-stage retrievers (dense, BM25) are built to be *cheap* and favour recall:
get the right chunk somewhere in the top 20–100. A **cross-encoder** then reads
each `(query, chunk)` pair *together* and scores relevance far more accurately —
too slow to run over the whole corpus, perfect over a shortlist.

```
retrieve top-N (cheap)  →  cross-encoder rerank  →  keep top-k (precise)
```


In [ ]:
# --- setup: make `import ragkit` work from notebooks/ or solutions/ ---
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
from ragkit.corpus import load_documents, load_corpus, load_qrels, tokenize
from ragkit.embed import get_embedder
from ragkit import llm

In [ ]:
from ragkit.reference import structure_aware_chunks, chunk_corpus
docs = load_documents()
chunks = chunk_corpus(docs, structure_aware_chunks)
emb = get_embedder()
mat = emb.encode([c.text for c in chunks])

def retrieve(query, n=10):
    qv = emb.encode(query)
    order = np.argsort(-(mat @ qv))[:n]
    return [chunks[i] for i in order]

### Exercise — two-stage retrieve-then-rerank

`get_cross_encoder()` returns a model with `.predict([(query, passage), ...])`
giving a relevance score per pair. Retrieve `n` candidates, score them, and
return the top `k` chunks by cross-encoder score.


In [ ]:
from ragkit.embed import get_cross_encoder
reranker = get_cross_encoder()

def rerank(query, n=10, k=3):
    candidates = retrieve(query, n)
    # YOUR CODE HERE:
    #   pairs  = [(query, c.text) for c in candidates]
    #   scores = reranker.predict(pairs)
    #   return the k candidates with the highest scores (highest first)
    ...

q = "how quickly must someone be paged for a customer-facing outage"
top = rerank(q, n=10, k=3)
assert len(top) == 3
assert all(hasattr(c, "chunk_id") for c in top)
print("reranked top-3:", [c.chunk_id for c in top])

### Watch it fix an ordering

Compare the rank of the truly-relevant chunk before vs after reranking. The
cross-encoder typically promotes the exact-answer chunk that first-stage cosine
buried behind topically-similar neighbours.


In [ ]:
def dense_order(query, n=10):
    return retrieve(query, n)

q = "how quickly must someone be paged for a customer-facing outage"
before = [c.chunk_id for c in dense_order(q, 10)]
after = [c.chunk_id for c in rerank(q, n=10, k=10)]
gold_doc = "incident-response"
def first_hit(order):
    return next((r for r, cid in enumerate(order, 1) if cid.split('#')[0] == gold_doc), None)
print("rank of incident-response chunk  before:", first_hit(before), " after:", first_hit(after))

Reranking is usually the single best precision upgrade after hybrid retrieval,
and it's a drop-in: retrieve wide, rerank, keep few.
